## 단일표본 t검정 ##

- 표본 데이터 로딩
- 표본 데이터가 정규성 검정하기
- 단일표본 t검정
	- p-value < 0.05: 귀무가설 기각, 대립가설 채택, 통계적으로 유의미하다고 해석함
	- p-value >= 0.05: 귀무가설 채택, 대립가설 기각

### 정규성 검정하기 ###

In [25]:
# data 불러오기
with open("./datas2/성인여성_키_데이터.txt", "r") as f:
	data = f.read().split('\n')
	# print(data)
	data = list(map(float,data))

In [26]:
from scipy import stats
import numpy as np

# 평균과 표준편차 계산
mean_value = np.mean(data)
std_dev = np.std(data, ddof=0)  # 모집단 표준편차(기본 설정)

mean_value, std_dev

(np.float64(156.9332), np.float64(9.974187774450611))

### 가설 설정 ###
- 귀무 가설: 표본 데이터의 평균은 163과 같다.
- 대립 가설: 표본 데이터의 평균은 163과 다르다.

#### 정규성 검정 ####
- 방법1: KS-test

In [27]:
# 정규성 검정, KS-test(Kolmogorov-Smirnov) 검정
#  KS-test는 주어진 표본 데이터가 특정 이론적인 분포(예: 정규분포)를 따르는지 검정하는 방법

# 'norm': 비교 대상이 되는 이론적인 분포를 지정 ('norm'은 정규분포) -> SciPy 1.18.0 버전부터 stats.norm.cdf 로 변경
# args=(np.mean(data), np.std(data, ddof=0)): 모딥단 데이터의 평균과 표준편차를 사용하여 정규분포의 모수 설정
# args=(np.mean(data), np.std(data, ddof=1)): 표본 데이터의 평균과 표준편차를 사용하여 정규분포의 모수 설정
ks_statistic, p_value = stats.kstest(
  data, 
  stats.norm.cdf, 
  args=(np.mean(data), np.std(data, ddof=1)))
ks_statistic, p_value

(np.float64(0.11216041431631618), np.float64(0.8772092057322225))

In [28]:
stats.shapiro(data) 

ShapiroResult(statistic=np.float64(0.9535807148228921), pvalue=np.float64(0.3014257453674708))

### 검정하기

In [29]:
# 단일 표본 t 검정 수행
# 검중 수치 = 163, 귀무가설 평균키는 163이다.

print(stats.ttest_1samp(data, 163)) 

TtestResult(statistic=np.float64(-2.979804412662668), pvalue=np.float64(0.00651044533584796), df=np.int64(24))


## 독립 표본 t 검정 ##

In [30]:
import pandas as pd
import numpy as np

In [31]:
df1 = pd.read_csv("./datas2/반별_점수_type1.csv", encoding = "euc-kr") # encoding이 뭘로 되어 있는지 아는 것이 중요
# df1 = pd.read_csv("./datas2/반별_점수_type1.csv")
df1.head()

,반,점수
0,A,73
1,A,69
2,A,71
3,A,71
4,A,73


In [32]:
# A반, B반 데이터 분리
df1['점수'].loc[df1['반']=='A'].values
group_A = df1['점수'].loc[df1['반'] == 'A'].values

In [33]:
df1['점수'].loc[df1['반']=='B']
group_B = df1['점수'].loc[df1['반'] == 'B'].values

## 정규성 검정
- 정규성 검정 가설수립
	- 영가설 : 두 그룹의 표본 데이터는 정규 분포를 따른다
	- 대립가설 : 두 그룹의 표본 데이터는 정규성을 따르지 않는다

In [35]:
print(stats.kstest(
  group_A, 
  stats.norm.cdf, 
  args=(np.mean(group_A), np.std(group_A, ddof=0))))

KstestResult(statistic=np.float64(0.1290433386337495), pvalue=np.float64(0.8515822805406548), statistic_location=np.float64(73.0), statistic_sign=np.int8(1))


[해석]
- p-value >= 0.05 : 귀무가설을 기각하지 않음 정규성을 따른다고 볼 수 있다.

## 등분산성 검정
- 가설 수립
	- 영가설(귀무가설): 두 그룹은 분산이 유사하다 (등분산성)
	- 대립가설: 두 그룹은 분산이 유사하지 않다 (등분산을 따르지 않는다)

In [36]:
stats.levene(group_A, group_B)

LeveneResult(statistic=np.float64(2.033067087400979), pvalue=np.float64(0.1649640862221014))

[해석]
p-value >= 0.05 : 두 그룹은 등분산성을 따른다.

## 독립 표본 t-검정수행

- 가설 수립
	- 영가설(귀무가설) : 두 반의 표본의 평균은 같다.
	- 대립가설 : 두 반의 표본의 평균은 다르다.

In [37]:
stats.ttest_ind(group_A, group_B, equal_var=True)

TtestResult(statistic=np.float64(2.5128526794964134), pvalue=np.float64(0.018010953528937678), df=np.float64(28.0))

[해석]
- p-value < 0.05 : 귀무가설을 기각함, 대립가설 채택
- 두 반의 표본의 평균은 다르다는 것이 통계적으로 유의미하다